# IEEEP370 去嵌入## 目录* [单端](#Single-ended)  * [2x短路/开路/负载/直通、DUT 和夹具-DUT-夹具的单端仿真](#Single-Ended-simulation-of-2xThru,-DUT-and-Fixture-DUT-Fixture)  * [单端散射参数质量检查](#Single-Ended-S-parameters-quality-checking)  * [不进行阻抗校正的 IEEEP370_SE_NZC_2xthru](#IEEEP370_SE_NZC_2xthru-without-impedance-correction)  * [进行阻抗校正的 IEEEP370_SE_ZC_2xthru](#IEEEP370_SE_ZC_2xThru-with-impedance-correction)  * [与 AICC 去嵌入工具的单端比较](#Single-Ended-Comparison-with-AICC-De-Embedding-Utility)* [混合模式](#Mixed-mode)  * [2x短路/开路/负载/直通、DUT 和夹具-DUT-夹具的混合模式仿真](#Mixed-mode-simulation-of-2xThru,-DUT-and-Fixture-DUT-Fixture)  * [混合模式散射参数质量检查](#Mixed-mode-S-parameters-quality-checking)  * [不进行阻抗校正的 IEEEP370_MM_NZC_2xthru](#IEEEP370_MM_NZC_2xThru-without-impedance-correction)  * [进行阻抗校正的 IEEEP370_MM_ZC_2xthru](#IEEEP370_MM_ZC_2xThru-with-impedance-correction)  * [与 AICC 去嵌入工具的混合模式比较](#Mixed-mode-comparison-with-AICC-De-Embedding-Utility)## 简介[IEEE 370-2020 标准](https://standards.ieee.org/ieee/370/6165/)（《高达 50 GHz 频率的印刷电路板和相关互连的电气特性标准》）提供了良好的实践方法，以确保高达 50 GHz 频率的高频电气互连的测量数据质量。它推荐了确保高达 50 GHz 频率信号的测量数据准确性和一致性的方法和流程，特别是用于消除测试夹具和仪表的影响。在下面的示例中，并遵循标准中关于标签的建议，一个 2 端口 DUT 的每个端口都连接了一个夹具。连接到单端 DUT 端口 1 的夹具标记为 FIX-1。连接到单端 DUT 端口 2 的夹具标记为 FIX-2。DUT 和两个端部的夹具的级联结构被命名为复合结构或 FIX-DUT-FIX 结构。<img src="fix-dut-fix.svg" width="400"/>两个夹具的组合，例如 FIX-1 连接到 FIX-2，称为 FIX-FIX 结构或 *2x-短路/开路/负载/直通*。<img src="fix-fix.svg" width="270"/>此笔记本的目的在于说明 `scikit-rf` 对 IEEE P370 标准中提出的去嵌入方法进行的实现，以消除测试夹具的影响。

首先，让我们进行必要的 Python 导入：

In [ ]:
import matplotlib.pyplot as plt
from scipy.constants import c, pi

import skrf as rf
from skrf.calibration.deembedding import (
    IEEEP370,
    IEEEP370_FD_QM,
    IEEEP370_FER,
    IEEEP370_TD_QM,
    IEEEP370_MM_NZC_2xThru,
    IEEEP370_MM_ZC_2xThru,
    IEEEP370_SE_NZC_2xThru,
    IEEEP370_SE_ZC_2xThru,
)
from skrf.media import DefinedGammaZ0, MLine
from skrf.network import concat_ports

rf.stylely()

## 单端 <a class="anchor" id="Single-ended"></a>这些算法可以用于 2 端口单端网络。### 2x 直通、DUT（测试对象）和夹具-DUT-夹具的单端仿真 <a class="anchor" id="Single-Ended-simulation-of-2xThru,-DUT-and-Fixture-DUT-Fixture"></a>我们使用 `scikit-rf` 的 `MLine` 介质来模拟微带线伪影。这是一种方便的方法，可以在不离开 Python 环境或运行其他软件的情况下生成合成的散射参数网络。* `dut` 是一个 Beatty 结构，其左侧和右侧连接着两个宽度为 1xWidth 的均匀微带线，中间是一个宽度为 3xWidth 的微带线段。* `fdf` 是 FIX-DUT-FIX，即在 DUT 的左侧和右侧延伸一段微带线和一个平面到同轴连接器。* `s2xthru` 是 FIX-FIX，它是两个延伸线和连接器的级联，不包括 DUT。为了演示目的，线条的宽度改变了 8%，以显示阻抗失配对去嵌入过程的影响。目标是获得左侧和右侧夹具的模型，并通过从 FIX-DUT-FIX 中去除夹具的影响来提取 DUT。<img src="ieeep370deembedding/2x-Thru.svg" width="700"/><img src="ieeep370deembedding/FIX-DUT-FIX.svg" width="800"/>这些微带线的设计灵感来自 [这个示例](./Time domain reflectometry, measurement vs simulation.ipynb)。

In [ ]:
directory = 'ieeep370deembedding/'
freq = rf.Frequency(10e-3, 10, 1000, 'GHz')
W   = 3.00e-3
H   = 1.55e-3
T   = 50e-6
ep_r = 4.459
tanD = 0.0183
f_epr_tand = 1e9

# microstrip segments
MSL1 = MLine(frequency=freq, z0_port=50, w=W, h=H, t=T,
        ep_r=ep_r, mu_r=1, rho=1.712e-8, tand=tanD, rough=0.15e-6,
        f_low=1e3, f_high=1e12, f_epr_tand=f_epr_tand,
        diel='djordjevicsvensson', disp='kirschningjansen')

# capacitive 3 x width Beatty structure
MSL2 = MLine(frequency=freq, z0_port=50, w=3*W, h=H, t=T,
        ep_r=ep_r, mu_r=1, rho=1.712e-8, tand=tanD, rough=0.15e-6,
        f_low=1e3, f_high=1e12, f_epr_tand=f_epr_tand,
        diel='djordjevicsvensson', disp='kirschningjansen')

# microstrip segment with a 10% variation of width
MSL3 = MLine(frequency=freq, z0_port=50, w=0.92*W, h=H, t=T,
        ep_r=ep_r, mu_r=1, rho=1.712e-8, tand=tanD, rough=0.15e-6,
        f_low=1e3, f_high=1e12, f_epr_tand=f_epr_tand,
        diel='djordjevicsvensson', disp='kirschningjansen')

# connector SMA edge PCB mount soldered
CONN = DefinedGammaZ0(frequency=freq, gamma = 1j* 2 * pi * freq.f / c, z0_port = 50)
z_conn = 53.2    # ohm, connector impedance
d_conn = 50      # ps,   connector discontinuity delay
connector = CONN.line(50, 'ps', z0 = z_conn)

# building DUT
dut =    MSL1.line(20e-3, 'm') \
      ** MSL2.line(20e-3, 'm') \
      ** MSL1.line(20e-3, 'm')
dut.name = 'DUT'

# building FIXTURE-DUT-FIXTURE
thru1 = MSL1.line(25e-3, 'm')
thru3 = MSL3.line(25e-3, 'm')
fdf     = connector ** thru1 ** dut ** thru1 ** connector
fdf.name = 'FIX-DUT-FIX'

# building FIXTURE-FIXTURE with a 8% width variation from FIXTURE-DUT-FIXTURE(2xthru)
s2xthru = connector ** thru3 ** thru3 ** connector
s2xthru.name = '2x-Thru'

# extrapolate to DC for time step
dut_dc = IEEEP370.extrapolate_to_dc(dut)
fdf_dc = IEEEP370.extrapolate_to_dc(fdf)
s2xthru_dc = IEEEP370.extrapolate_to_dc(s2xthru)
# set True to write .s2p files
if False:
    s2xthru.write_touchstone(directory + 'se_2xthru_nonoise.s2p')
    fdf.write_touchstone(directory + 'se_fdf_nonoise.s2p')

在下面的图中，由于夹具引起的 DUT 的时移可以在 FIX-DUT-FIX 曲线中观察到。2x-Thru 曲线显示了与 FIX-DUT-FIX 预期的阻抗失配。连接器在 2x-Thru 和 FIX-DUT-FIX 阻抗阶跃响应的末端增加了阻抗峰值。参考 DUT 仅为 Beatty 结构，不包含夹具。`_dc` 网络被外推到 DC 点，用于时域阶跃曲线。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))

axs[0].set_title('Time Step')
dut_dc.plot_z_time_step(0, 0, ax = axs[0])
fdf_dc.plot_z_time_step(0, 0, ax = axs[0])
s2xthru_dc.plot_z_time_step(0, 0, ax = axs[0])
axs[0].set_xlim((-1.5, 2))
axs[0].legend(loc = 'lower left')

axs[1].set_title('Frequency')
dut.plot_s_db(0, 0, ax = axs[1])
fdf.plot_s_db(0, 0, ax = axs[1])
s2xthru.plot_s_db(0, 0, ax = axs[1])
axs[1].legend(loc = 'lower left')
fig.tight_layout()

### 单端散射参数质量检查

首先，检查数据的质量指标，以确保其不违反互易性、倒易性和因果性。在频域中进行的初步质量检查会得到百分比评分，仅供参考。结果会被评估为“良好”、“可接受”、“结果不明确”或“差”。结果不明确或差可能表明需要重新校准 VNA 并重新进行测量。此快速检查可能会发现潜在问题，从而对测量、仿真或去嵌入过程提出质疑。

In [ ]:
fd_qm = IEEEP370_FD_QM()
print(s2xthru.name)
qm_2xthru = fd_qm.check_se_quality(s2xthru)
fd_qm.print_qm(qm_2xthru)
print(fdf.name)
qm_fdf = fd_qm.check_se_quality(fdf)
fd_qm.print_qm(qm_fdf)

所有结果都属于最佳等级，这对于合成数据来说并不令人惊讶。通过使用 `verbose = True` 参数，可以深入了解检查过程。因果性检查验证复数散射参数在复平面上是否以顺时针方向旋转。无源性检查验证在每个频率下，散射参数的 2-范数是否小于或等于 1。互易性检查计算 $S_{ij}$ 和 $S_{ji}$ 对之间的绝对差值。有关更多详细信息，请参考 IEEE 370-2020。

In [ ]:
qm_2xthru = fd_qm.check_se_quality(s2xthru, verbose = True)

基于应用的时间域质量检测，针对具有指定数据速率和上升时间要求的互连应用。这是一个高级过程，旨在根据输入数据和理想重构数据之间的峰值失真分析，提供基于毫伏级的物理估计。如果需要，原始的散射参数将被外推到所需数据速率的三倍频率。然后，因果、无源和互易模型将被重建，并通过脉冲进行激励。原始外推信号与时间域中的理想响应之间的差异会在单位时间间隔内进行积分，从而得到以毫伏为单位的指标。结果将被评估为“良好”、“可接受”、“无法确定”或“差”。

In [ ]:
td_qm = IEEEP370_TD_QM(1e9, # bps
                       32,  # unit interval samples number
                       0.4, # rise-time from 20% to 80% in fraction of width
                       1,   # gaussian pulse
                       1,   # constant extrapolation
                       verbose = False)
print(s2xthru.name)
qm_2xthru = td_qm.check_se_quality(s2xthru)
td_qm.print_qm(qm_2xthru)
print(fdf.name)
qm_fdf = td_qm.check_se_quality(fdf)
td_qm.print_qm(qm_fdf)

所有结果都属于最佳等级，这对于合成数据来说并不令人惊讶。通过使用 `verbose = True` 参数，可以深入了解检查过程。绘制了原始模型和强制因果关系模型的外推、脉冲和时域响应。更多详细信息，请参考 IEEE 370-2020。

In [ ]:
qm_2xthru = td_qm.check_se_quality(s2xthru, verbose = True)

在应用去嵌入算法之前，必须验证输入数据是否符合 IEEE 370 夹具电气要求 (FER)。

In [ ]:
fer = IEEEP370_FER()
fig = fer.plot_fd_se_fer(s2xthru)

In [ ]:
fig = fer.plot_td_se_fer(s2xthru, fdf)

2x-Thru 的性能被认为符合 B 级标准，适用于低于 10 GHz 的频率。这是因为 2x-Thru 和 FIX-DUT-FIX 之间存在人为的阻抗失配，导致其未达到 FER5 A 限制。

### IEEEP370_SE_NZC_2xthru，不进行阻抗校正 <a class="anchor" id="IEEEP370_SE_NZC_2xthru-without-impedance-correction"></a>此方法将 2x-Thru 作为输入。它是一种简单有效的二分算法，无法校正 FIX-FIX 和 FIX-DUT-FIX 之间的阻抗差异。应尽量减小这种不需要的差异。它可能由于材料的不均匀性、制造工艺或如果器件未构建在同一电路板上而出现。S 参数的二分是通过对 S11 和 S22 进行时域门控、对通过回波损耗校正后的 S21 取适当的平方根，并根据夹具信号流程图重新组合参数来完成的。此方法给出的结果比较粗略，但具有较强的鲁棒性。IEEE 370 建议进行以下一致性测试：* 对 2x-Thru 进行自去嵌入处理，使残余插入损耗的绝对值小于 ±0.1 dB，残余相位小于 ±1°。* 将夹具模型的 TDR 与 FIX-DUT-FIX 进行比较。

In [ ]:
dm_nzc = IEEEP370_SE_NZC_2xThru(dummy_2xthru = s2xthru, name = '2xthru')
nzc_fix1 = dm_nzc.s_side1
nzc_fix1.name = 'nzc_FIX-1'
nzc_fix2 = dm_nzc.s_side2
nzc_fix2.name = 'nzc_FIX-2'
nzc_d_dut = dm_nzc.deembed(fdf)
nzc_d_dut.name = 'nzc_DUT'
nzc_fix1_dc = IEEEP370.extrapolate_to_dc(nzc_fix1)
nzc_d_dut_dc = IEEEP370.extrapolate_to_dc(nzc_d_dut)

fig, ax = dm_nzc.plot_check_residuals()

如图所示，夹具模型与 2x 直通数据的吻合度非常好，残差的幅度和相位远小于 IEEEP370 ±0.1 dB 和 ±1° 的限制。

In [ ]:
fig, ax = dm_nzc.plot_check_impedance(fdf)

两种测试夹具在从起点到中间的2x-Thru测量中都显示出非常一致的结果。但是，在2x-Thru和FIX-DUT-FIX之间添加的5%人造阻抗失配会降低DUT去嵌入性能。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].set_title('Time Step')
fdf_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dotted', color = 'm')
nzc_d_dut_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'r')
dut_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dashed', color = 'k')
axs[0].set_xlim((-2, 2))
axs[0].legend(loc = 'lower left')

axs[1].set_title('Frequency')
fdf.plot_s_db(0, 0, ax = axs[1], color = 'm', linestyle = 'dotted')
nzc_d_dut.plot_s_db(0, 0, ax = axs[1], color = 'r')
dut.plot_s_db(0, 0, ax = axs[1], linestyle = 'dashed', color = 'k')
axs[1].set_ylim((-40, 5))
fig.tight_layout()

NZC 去嵌入算法已移除了夹具引起的延迟。正如预期的那样，2x-Thru 和 FIX-DUT-FIX 之间的阻抗失配会降低去嵌入效果，导致在时域中时间零点之前出现阻抗反射。这在频域中也表现为波纹。

### IEEEP370_SE_ZC_2xThru，带有阻抗校正 <a class="anchor" id="IEEEP370_SE_ZC_2xThru-with-impedance-correction"></a>此方法将 2x-Thru 和 FIX-DUT-FIX 作为输入。该算法通过在时域传输中将 2x-Thru 的延迟减半来计算夹具的长度。传播常数 `gamma` 也会从 2x-Thru 中确定。然后，它以迭代的方式，以确定起始阻抗和去除单个时域样本长度的传输线为周期，去除 FIX-DUT-FIX 时域阻抗曲线。由于 2x-Thru 仅用于确定长度和传播常数，因此 FIX-DUT-FIX 与其之间的阻抗失配的影响得以降低。如果需要使用相同的夹具去除另一个 FIX-DUT-FIX，则两个 FIX-DUT-FIX 之间的阻抗失配会对去除性能产生不利影响。与 `IEEEP370_SE_NZC_2xThru` 2x-Thru 散射参数二分法相比，此方法更高级。它具有更多的选项，并且通常可以提供更好的结果，但需要更多的处理能力。

In [ ]:
dm_zc  = IEEEP370_SE_ZC_2xThru(dummy_2xthru = s2xthru, dummy_fix_dut_fix = fdf,
                         bandwidth_limit = 10e9, pullback1 = 0, pullback2 = 0,
                         leadin = 0, NRP_enable = False,
                         name = 'zc2xthru')
zc_d_dut = dm_zc.deembed(fdf)
zc_d_dut.name = 'zc_DUT'
zc_fix1 = dm_zc.s_side1
zc_fix1.name = 'zc_FIX-1'
zc_fix2 = dm_zc.s_side2
zc_fix2.name = 'zc_FIX-2'
zc_fix1_dc = IEEEP370.extrapolate_to_dc(zc_fix1)
zc_d_dut_dc = IEEEP370.extrapolate_to_dc(zc_d_dut)

fig, ax = dm_zc.plot_check_residuals()

从残差的幅度和相位小于 IEEE 370 ±0.1 dB 和 ±1° 的限制可以看出，夹具模型与 2x 直通之间的吻合度很好。与 NZC 算法相比，残差更高。这是预期的结果，因为夹具是基于 FIX-DUT-FIX 阻抗模型，该模型人为地增加了与 2x 直通之间的阻抗失配。

In [ ]:
fig, ax = dm_zc.plot_check_impedance()

这两个夹具从开始到中间都显示出与 FIX-DUT-FIX 结果的高度一致性。2x-Thru 仅用于确定夹具长度和传播常数 gamma。如果需要捕获在时间零点之前的多个非参考阻抗样本，则 `leadin` 选项非常有用。`pullback1` 和 `pullback2` 选项可以将夹具长度缩短为整数个时间样本。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].set_title('Time Step')
fdf_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dotted', color = 'm')
zc_d_dut_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'r')
dut_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dashed', color = 'k')
axs[0].set_xlim((-1, 2))
axs[0].set_ylim((15, 55))
axs[0].legend(loc = 'lower left')

axs[1].set_title('Frequency')
fdf.plot_s_db(0, 0, ax = axs[1], color = 'm', linestyle = 'dotted')
zc_d_dut.plot_s_db(0, 0, ax = axs[1], color = 'r')
dut.plot_s_db(0, 0, ax = axs[1], linestyle = 'dashed', color = 'k')
axs[1].set_ylim((-40, 5))
fig.tight_layout()

如预期，在时域和频域中，ZC去嵌入算法与参考DUT的匹配程度均优于NZC算法。这是因为，在本示例中，它不受2xThru和FIX-DUT-FIX之间的阻抗失配的影响。在实际设计中，应尽可能减小这种阻抗差异。

### 使用 AICC 去嵌入工具进行单端比较 <a class="anchor" id="Single-Ended-Comparison-with-AICC-De-Embedding-Utility"></a>提供了一组参考的 Matlab 或 Octave 代码，用于实现 IEEE P370 NZC 和 ZC 去嵌入算法，并采用开源 BSD-3-Clause 许可协议提供 [在 IEEE 仓库中](https://opensource.ieee.org/elec-char/ieee-370/-/tree/master/TG1)。然而，并非所有人都能访问 Matlab 和 RF 工具箱。或许，这就是您阅读本文的原因，期待使用 `scikit-rf` 和 Python。一个带有 GUI 界面的 Matlab 编译二进制文件，名为“ACS 去嵌入工具”，可在 [Amphenol 网站](https://www.amphenol-cs.com/software) 上找到。<img src="ieeep370deembedding/AICC_Deembedding.png">让我们比较该工具与 `scikit-rf` 中去嵌入算法的输出结果，以进行一致性检查。

In [ ]:
# read AICC generated files
nzc_ref = rf.Network(directory + 'deembedded_SE_NZC_se_fdf.s2p')
nzc_ref.name = 'aicc_nzc'
zc_ref = rf.Network(directory + 'deembedded_SE_ZC_se_fdf.s2p')
zc_ref.name = 'aicc_zc'
nzc_ref_dc = IEEEP370.extrapolate_to_dc(nzc_ref)
zc_ref_dc = IEEEP370.extrapolate_to_dc(zc_ref)

# make purposely wrong NZC side 2 flip for comparison with AICC Tool
nzc_wrong_d_dut = dm_nzc.s_side1.inv ** fdf ** dm_nzc.s_side2.inv
nzc_wrong_d_dut.name = 'wrong flip'
nzc_wrong_d_dut_dc = IEEEP370.extrapolate_to_dc(nzc_wrong_d_dut)

# compute absolute differences
# division of complex reflexion coeff is like subtraction in dB and deg
delta_nzc = nzc_ref / nzc_wrong_d_dut
delta_zc = zc_ref / zc_d_dut

# plot them all
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].set_title('AICC Tool NZC side 1')
nzc_wrong_d_dut_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'r')
nzc_d_dut_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'g')
nzc_ref_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dashed', color = 'k')
axs[0].legend(loc = 'lower right')
axs[0].set_xlim((-1, 2))

axs[1].set_title('AICC Tool NZC side 2')
nzc_wrong_d_dut_dc.plot_z_time_step(1, 1, ax = axs[1], color = 'r')
nzc_d_dut_dc.plot_z_time_step(1, 1, ax = axs[1], color = 'g')
nzc_ref_dc.plot_z_time_step(1, 1, ax = axs[1], linestyle = 'dashed', color = 'k')
axs[1].legend(loc = 'lower right')
axs[1].set_xlim((-1, 2))

plt.tight_layout()

在比较 `scikit-rf` 和 `AICC` 的结果时，NZC 算法在 DUT 部分之后显示出轻微的差异（绿色曲线）。这是因为 `AICC` FIX-2 的翻转操作有误，这可以通过对称的 FIX-DUT-FIX 结构的侧面 1 和侧面 2 的结果差异来证明（虚线黑色曲线）。如果将此错误传递到 `scikit-rf` 中进行比较，则结果会完全一致（红色曲线）。使用的 `AICC` 版本为 1.0.0。

In [ ]:
# plot them all
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].set_title('AICC Tool NZC side 1')

axs[0].set_title('AICC Tool ZC side 1')
zc_d_dut_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'r')
zc_ref_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dashed', color = 'k')
axs[0].legend(loc = 'lower right')
axs[0].set_xlim((-1, 2))

axs[1].set_title('AICC Tool NZC side 2')
zc_d_dut_dc.plot_z_time_step(1, 1, ax = axs[1], color = 'r')
zc_ref_dc.plot_z_time_step(1, 1, ax = axs[1], linestyle = 'dashed', color = 'k')
axs[1].legend(loc = 'lower right')
axs[1].set_xlim((-1, 2))

plt.tight_layout()

在比较 `scikit-rf` 和 `AICC` 的结果时，从时域上看，ZC 对两侧的拟合效果都非常完美。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].set_title('Absolute magnitude difference')
delta_nzc.plot_s_db(1,0, ax = axs[0])
delta_zc.plot_s_db(1,0, ax = axs[0])
axs[0].legend(loc = 'upper right')

axs[1].set_title('Absolute phase difference')
delta_nzc.plot_s_deg(1,0, ax = axs[1])
delta_zc.plot_s_deg(1,0, ax = axs[1])
axs[1].legend(loc = 'upper right')

plt.tight_layout()

对于研究的单端情况，`scikit-rf` 提供的结果与 IEEE 370 去嵌入例程的 `AICC De-embedding Utility` 实现保持一致。此外，它还解决了 `AICC De-embedding Utility` 1.0.0 版本中的 NZC FIX-2 极性错误。NZC（包含极性错误）和 ZC 的幅度和相位绝对差值均在 ±0.06 dB 和 ±0.4° 以内，最高频率为 10 GHz，这是一个非常好的结果。

## 混合模式ZC 和 NZC 去嵌入算法与差分 4 端口网络兼容。使用单端到广义混合模式的转换，以分离差分和共模子网络，并对夹具进行单独建模。然后，将差分和共模夹具合并并转换回 4 端口单端网络。这种方法假定差分到共模的转换可以忽略不计。

### 2x直通、DUT 和夹具-DUT-夹具的混合模式仿真 <a class="anchor" id="Mixed-mode-simulation-of-2xThru,-DUT-and-Fixture-DUT-Fixture"></a>我们使用 [Qucs](http://qucs.sourceforge.net/) 来模拟耦合微带线结构。这是一个免费的仿真器，可以生成基于公式的射频器件的 S 散射参数，以及其他功能。* `diff_dut` 是一种 Beatty 结构，包含一个 3 倍宽度的耦合微带线段，其左侧和右侧分别连接两个均匀的 1 倍宽度的耦合微带线。* `diff_fdf` 是 FIX-DUT-FIX，它通过一个耦合微带线和一个平面到同轴连接器，将 DUT 向左和向右延伸。* `diff_2xthru` 是 FIX-FIX，它是两个延伸线和连接器的串联，不包含 DUT。为了演示，我们改变了线的宽度，以展示阻抗失配对去嵌入过程的影响。目标是获取左侧和右侧夹具的模型，并通过从 FIX-DUT-FIX 中去除夹具的影响来提取 DUT。文件和 `qucs` 源代码位于与此笔记本相同的目录 `ieeep370deembedding` 中。<img src="ieeep370deembedding/diff_fdf.png">

在下图所示，由夹具引起的DUT的时间延迟在FIX-DUT-FIX的曲线中可见。2x-Thru曲线显示了与FIX-DUT-FIX的预期阻抗失配。连接器在2x-Thru和FIX-DUT-FIX阻抗阶跃响应的末端增加了阻抗峰值。参考DUT仅为Beatty结构，不包含夹具。在这种耦合线几何结构中，共模阻抗与25欧姆的阻抗差异很大。

In [ ]:
# load single-ended data
se_dut = rf.Network(directory + 'diff_dut.s4p')
se_2xthru = rf.Network(directory + 'diff_2xthru_with_connectors.s4p')
se_2xthru.name = 'diff_2xthru'
#se_2xthru = se_2xthru.delay(10, 'ps', 3) # skew
se_fdf = rf.Network(directory + 'diff_fdf_with_connectors.s4p')
se_fdf.name = 'diff_fdf'
#se_fdf = se_fdf.delay(10, 'ps', 0) # skew

# transform to mixed-modes
mm_dut = se_dut.copy()
mm_dut.se2gmm(p = 2)
mm_2xthru = se_2xthru.copy()
mm_2xthru.se2gmm(p = 2)
mm_fdf = se_fdf.copy()
mm_fdf.se2gmm(p = 2)

# extrapolate to DC for time step
mm_dut_dc = IEEEP370.extrapolate_to_dc(mm_dut)
mm_2xthru_dc = IEEEP370.extrapolate_to_dc(mm_2xthru)
mm_fdf_dc = IEEEP370.extrapolate_to_dc(mm_fdf)

# set True to write .s4p files
if False:
    connectors = concat_ports([connector, connector], port_order = 'second')
    se_2xthru = connectors ** rf.Network(directory + 'diff_2xthru.s4p') ** connectors
    se_fdf = connectors ** rf.Network(directory + 'diff_fdf.s4p') ** connectors
    se_2xthru.write_touchstone(directory + 'diff_2xthru_with_connectors.s4p')
    se_fdf.write_touchstone(directory + 'diff_fdf_with_connectors.s4p')

# looking at the newtorks
fig, axs = plt.subplots(1, 2, figsize=(8, 4))

axs[0].set_title('Time Step')
mm_dut_dc.plot_z_time_step(0, 0, ax = axs[0])
mm_fdf_dc.plot_z_time_step(0, 0, ax = axs[0])
mm_2xthru_dc.plot_z_time_step(0, 0, ax = axs[0])
mm_dut_dc.plot_z_time_step(2, 2, ax = axs[0])
mm_fdf_dc.plot_z_time_step(2, 2, ax = axs[0])
mm_2xthru_dc.plot_z_time_step(2, 2, ax = axs[0])
axs[0].set_xlim((-2, 2))
axs[0].legend(loc = 'center left')

axs[1].set_title('Frequency')
mm_dut.plot_s_db(0, 0, ax = axs[1])
mm_fdf.plot_s_db(0, 0, ax = axs[1])
mm_2xthru.plot_s_db(0, 0, ax = axs[1])
axs[1].legend(loc = 'lower left')

fig.tight_layout()

### 混合模式散射参数质量检查

首先，检查数据的质量指标，以确保它们不违反能量守恒、互易性和因果性。在频域中进行的初步质量检查会产生百分比得分，仅供参考。结果会被评估为“良好”、“可接受”、“不确定”或“差”。“不确定”或“差”的结果可能表明需要重新校准VNA并重新进行测量。此快速检查可能会发现潜在问题，从而对测量、仿真或去嵌入过程提出质疑。仅检查差分模式和共模。

In [ ]:
fd_qm = IEEEP370_FD_QM()
print(mm_2xthru.name)
qm_2xthru = fd_qm.check_mm_quality(se_2xthru)
fd_qm.print_qm(qm_2xthru)
print(mm_fdf.name)
qm_fdf = fd_qm.check_mm_quality(se_fdf)
fd_qm.print_qm(qm_fdf)

所有结果都属于最高等级，对于合成数据来说，这并不令人意外。

基于应用的时域质量检查针对具有指定数据速率和上升时间的互连应用。这是一个高级过程，旨在根据输入数据和理想重构数据之间的峰值失真分析，提供基于毫伏的物理估计。仅检查差分模式和共模。

In [ ]:
td_qm = IEEEP370_TD_QM(1e9, # bps
                       32,  # UI samples
                       0.4, # rise-time in fraction of UI
                       1,   # Gaussian pulse,
                       2,   # zero padding extrapolation
                       verbose = False)
print(mm_2xthru.name)
qm_2xthru = td_qm.check_mm_quality(se_2xthru)
td_qm.print_qm(qm_2xthru)
print(mm_fdf.name)
qm_fdf = td_qm.check_mm_quality(se_fdf)
td_qm.print_qm(qm_fdf)

所有结果都达到了最高等级，这对于合成数据来说并不令人意外。

在执行去嵌入算法之前，必须验证输入数据是否符合 IEEE 370 夹具电气要求 (FER)。

In [ ]:
fer = IEEEP370_FER()
fig = fer.plot_fd_mm_fer(se_2xthru)

In [ ]:
fig = fer.plot_td_mm_fer(se_2xthru, se_fdf)

在 10 GHz 以下的频率范围内，2x-Thru 的性能被认为符合 C 级标准。这是因为 2x-Thru 和 FIX-DUT-FIX 之间存在人为的阻抗失配，导致其未能满足 FER5 A 和 B 的限制。

### IEEEP370_MM_NZC_2xThru，不进行阻抗校正 <a class="anchor" id="IEEEP370_MM_NZC_2xThru-without-impedance-correction"></a>此方法以 2x-Thru 为输入。它是一种简单有效的二分法算法，无法校正 FIX-FIX 和 FIX-DUT-FIX 之间的阻抗差异。应尽量减小这种不需要的差异。它可能由于材料的不均匀性、制造工艺或如果器件未构建在同一电路板上而出现。S 参数的二分法是通过对 S11 和 S22 进行时序门控、对经过回波损耗校正后的 S21 进行适当的平方根运算以及根据夹具信号流图重新组合参数来完成的。此方法给出的结果相对粗糙，但具有很强的鲁棒性。IEEE 370 建议进行以下一致性测试：* 对 2x-Thru 进行自去嵌，使其残余插入损耗的绝对值为 < ±0.1 dB，残余相位为 < ±1°。* 将夹具模型的 TDR 与 FIX-DUT-FIX 进行比较。

In [ ]:
# de-embedding
z0 = 50
dm_mmnzc  = IEEEP370_MM_NZC_2xThru(dummy_2xthru = se_2xthru, z0 = z0, name = 'mmnzc')
mm_nzc_side1 = dm_mmnzc.se_side1.copy()
mm_nzc_side1.name = 'FIX-1-2'
mm_nzc_side1.se2gmm(p = 2)
se_d_dut_nzc = dm_mmnzc.deembed(se_fdf)
se_d_dut_nzc.name = 'd_dut_nzc'
mm_d_dut_nzc = se_d_dut_nzc.copy()
mm_d_dut_nzc.se2gmm(p = 2)
mm_nzc_side1_dc = IEEEP370.extrapolate_to_dc(mm_nzc_side1)
mm_d_dut_nzc_dc = IEEEP370.extrapolate_to_dc(mm_d_dut_nzc)

fig, ax = dm_mmnzc.plot_check_residuals()

如图所示，夹具模型与 2x 直通法测得的结果非常吻合，残差的幅度和相位远小于 IEEEP370 ±0.1 dB 和 ±1° 的限值。

In [ ]:
fig, ax = dm_mmnzc.plot_check_impedance(se_fdf)

两种测试夹具从开始到中间都显示出与2x-直通测试结果的高度一致性。但是，在2x-直通测试和FIX-DUT-FIX之间添加的人工阻抗失配会降低DUT（待测器件）去嵌入性能。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].set_title('Time Step')
mm_fdf_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dotted', color = 'm')
mm_d_dut_nzc_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'r')
mm_dut_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dashed', color = 'k')
mm_fdf_dc.plot_z_time_step(2, 2, ax = axs[0], linestyle = 'dotted', color = 'b')
mm_d_dut_nzc_dc.plot_z_time_step(2, 2, ax = axs[0], color = 'c')
mm_dut_dc.plot_z_time_step(2, 2, ax = axs[0], linestyle = 'dashed', color = 'k')
axs[0].set_xlim((-2, 2))
axs[0].legend(loc = 'center left')

axs[1].set_title('Frequency')
mm_fdf.plot_s_db(0, 0, ax = axs[1], color = 'm', linestyle = 'dotted')
mm_d_dut_nzc.plot_s_db(0, 0, ax = axs[1], color = 'r')
mm_dut.plot_s_db(0, 0, ax = axs[1], linestyle = 'dashed', color = 'k')
axs[1].set_ylim((-40, 5))
fig.tight_layout()

NZC 去嵌入算法已消除了夹具的延迟。正如预期的那样，2x-Thru 和 FIX-DUT-FIX 之间的阻抗失配会降低去嵌入效果，导致在时域中时间零点之前出现阻抗反射。这在频域中也表现为波纹。

### IEEEP370_MM_ZC_2xThru，带有阻抗校正 <a class="anchor" id="IEEEP370_MM_ZC_2xThru-with-impedance-correction"></a>此方法以 2x-Thru 和 FIX-DUT-FIX 作为输入。它对 FIX-FIX 和 FIX-DUT-FIX 线路之间的（不需要的）阻抗差异进行校正。该算法通过在时域传输中将 2x-Thru 的延迟减半来计算夹具的长度。传播常数 gamma 也是从 2x-Thru 中确定的。然后，它以迭代的方式，以确定起始阻抗和去除单个时域采样长度的传输线为周期，去除 FIX-DUT-FIX 时域阻抗曲线。与 IEEEP370_MM_NZC_2xThru 相比，IEEEP370_MM_ZC_2xThru 具有更多选项，并且通常能获得更好的结果，但它会消耗更多的处理能力。

In [ ]:
# de-embedding
dm_mmzc  = IEEEP370_MM_ZC_2xThru(dummy_2xthru = se_2xthru, dummy_fix_dut_fix = se_fdf,
                         bandwidth_limit = 10e9, pullback1 = 0, pullback2 = 0,
                         leadin = 0, NRP_enable = False, name = 'mmzc')
mm_zc_side1 = dm_mmzc.se_side1.copy()
mm_zc_side1.name = 'FIX-1-2'
mm_zc_side1.se2gmm(p = 2)
se_d_dut_zc = dm_mmzc.deembed(se_fdf)
se_d_dut_zc.name = 'd_dut_zc'
mm_d_dut_zc = se_d_dut_zc.copy()
mm_d_dut_zc.se2gmm(p = 2)
mm_zc_side1_dc = IEEEP370.extrapolate_to_dc(mm_zc_side1)
mm_d_dut_zc_dc = IEEEP370.extrapolate_to_dc(mm_d_dut_zc)

fig, ax = dm_mmzc.plot_check_residuals()

如图所示，夹具模型与 2x 直通的吻合度良好，残差差分模式的幅度和相位均小于 IEEE 370 ±0.1 dB 和 ±1° 的限制。共模残差在 ±0.3 dB 和 ±6° 范围内，虽然略高，但尚可接受。残差高于使用 NZC 算法时的残差。这是预期的结果，因为夹具的建模基于 FIX-DUT-FIX 阻抗模型，该模型人为地增加了与 2x 直通之间的阻抗失配。对于共模，这一点尤为明显。

In [ ]:
fig, ax = dm_mmzc.plot_check_impedance()

两种夹具在从起始到中间的范围内，都显示出与 FIX-DUT-FIX 结果的高度一致性。2x-Thru 仅用于确定夹具长度和传播常数 gamma。如果需要捕获在时间零点之前的多个非参考阻抗样本，则 `leadin` 选项非常有用。`pullback1` 和 `pullback2` 选项可以将夹具长度缩短一个整数倍的时间样本。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].set_title('Time Step')
mm_fdf_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dotted', color = 'm')
mm_d_dut_zc_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'r')
mm_dut_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dashed', color = 'k')
mm_fdf_dc.plot_z_time_step(2, 2, ax = axs[0], linestyle = 'dotted', color = 'b')
mm_d_dut_zc_dc.plot_z_time_step(2, 2, ax = axs[0], color = 'c')
mm_dut_dc.plot_z_time_step(2, 2, ax = axs[0], linestyle = 'dashed', color = 'k')
axs[0].set_xlim((-2, 2))
axs[0].legend(loc = 'center left')

axs[1].set_title('Frequency')
mm_fdf.plot_s_db(0, 0, ax = axs[1], color = 'm', linestyle = 'dotted')
mm_d_dut_zc.plot_s_db(0, 0, ax = axs[1], color = 'r')
mm_dut.plot_s_db(0, 0, ax = axs[1], linestyle = 'dashed', color = 'k')
axs[1].set_ylim((-40, 5))
fig.tight_layout()

正如预期的那样，在时域和频域中，ZC 去嵌入算法与参考 DUT 的一致性都优于 NZC。这是因为该算法不受本示例中 2xThru 和 FIX-DUT-FIX 之间的阻抗失配的影响。在实际设计中，应尽可能地减小这种阻抗差异。

### 使用 AICC 去嵌入工具进行混合模式比较让我们比较来自 `scikit-rf` 和 `AICC 去嵌入工具` 的去嵌入后的 DUT（被测设备），以完成本笔记本的演示。

In [ ]:
# load single-ended data
se_ref_nzc = rf.Network(directory + 'deembedded_MM_NZC_diff_fdf_with_connectors.s4p')
se_ref_nzc.name = 'aicc_nzc'
se_ref_zc = rf.Network(directory + 'deembedded_MM_ZC_diff_fdf_with_connectors.s4p')
se_ref_zc.name = 'aicc_zc'

# transform to mixed-modes
mm_ref_nzc = se_ref_nzc.copy()
mm_ref_nzc.se2gmm(p = 2)
mm_ref_zc = se_ref_zc.copy()
mm_ref_zc.se2gmm(p = 2)

# extrapolate to DC for time step
mm_ref_nzc_dc = IEEEP370.extrapolate_to_dc(mm_ref_nzc)
mm_ref_zc_dc = IEEEP370.extrapolate_to_dc(mm_ref_zc)

# make purposely wrong NZC side 2 flip for comparison with AICC Tool
se_nzc_wrong_d_dut = dm_mmnzc.se_side1.inv ** se_fdf ** dm_mmnzc.se_side2.inv
mm_nzc_wrong_d_dut = se_nzc_wrong_d_dut.copy()
mm_nzc_wrong_d_dut.se2gmm(p=2)
mm_nzc_wrong_d_dut.name = 'wrong flip'
mm_nzc_wrong_d_dut_dc = IEEEP370.extrapolate_to_dc(mm_nzc_wrong_d_dut)

# compute absolute differences
# division of complex reflexion coeff is like subtraction in dB and deg
delta_nzc = mm_ref_nzc / mm_nzc_wrong_d_dut
delta_zc = mm_ref_zc / mm_d_dut_zc

# plot them all
fig, axs = plt.subplots(1, 2, figsize=(8, 4))

axs[0].set_title('AICC Tool NZC side 1')
mm_nzc_wrong_d_dut_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'r')
mm_d_dut_nzc_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'g')
mm_ref_nzc_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dashed', color = 'k')
mm_nzc_wrong_d_dut_dc.plot_z_time_step(2, 2, ax = axs[0], color = 'b')
mm_d_dut_nzc_dc.plot_z_time_step(2, 2, ax = axs[0], color = 'c')
mm_ref_nzc_dc.plot_z_time_step(2, 2, ax = axs[0], linestyle = 'dashed', color = 'k')
axs[0].legend(loc = 'center right')
axs[0].set_xlim((-1, 2))

axs[1].set_title('AICC Tool NZC side 2')
mm_nzc_wrong_d_dut_dc.plot_z_time_step(1, 1, ax = axs[1], color = 'r')
mm_d_dut_nzc_dc.plot_z_time_step(1, 1, ax = axs[1], color = 'g')
mm_ref_nzc_dc.plot_z_time_step(1, 1, ax = axs[1], linestyle = 'dashed', color = 'k')
mm_nzc_wrong_d_dut_dc.plot_z_time_step(3, 3, ax = axs[1], color = 'b')
mm_d_dut_nzc_dc.plot_z_time_step(3, 3, ax = axs[1], color = 'c')
mm_ref_nzc_dc.plot_z_time_step(3, 3, ax = axs[1], linestyle = 'dashed', color = 'k')
axs[1].legend(loc = 'center right')
axs[1].set_xlim((-1, 2))

fig.tight_layout()

在比较 `scikit-rf` 和 `AICC` 的结果时，NZC 算法在 DUT 部分之后显示出轻微的差异（绿色曲线）。这是因为 `AICC` FIX-2 的翻转操作有误，这可以通过对称的 FIX-DUT-FIX 结构的侧面 1 和侧面 2 的结果差异来证明（虚线黑色曲线）。如果将此错误传播到 `scikit-rf` 中以便进行比较，则结果会完全一致（红色曲线）。使用的 `AICC` 版本是 1.0.0。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].set_title('AICC Tool NZC side 1')

axs[0].set_title('AICC Tool ZC side 1')
mm_d_dut_zc_dc.plot_z_time_step(0, 0, ax = axs[0], color = 'r')
mm_ref_zc_dc.plot_z_time_step(0, 0, ax = axs[0], linestyle = 'dashed', color = 'k')
mm_d_dut_zc_dc.plot_z_time_step(2, 2, ax = axs[0], color = 'c')
mm_ref_zc_dc.plot_z_time_step(2, 2, ax = axs[0], linestyle = 'dashed', color = 'k')
axs[0].legend(loc = 'center right')
axs[0].set_xlim((-1, 2))

axs[1].set_title('AICC Tool NZC side 2')
mm_d_dut_zc_dc.plot_z_time_step(1, 1, ax = axs[1], color = 'r')
mm_ref_zc_dc.plot_z_time_step(1, 1, ax = axs[1], linestyle = 'dashed', color = 'k')
mm_d_dut_zc_dc.plot_z_time_step(3, 3, ax = axs[1], color = 'c')
mm_ref_zc_dc.plot_z_time_step(3, 3, ax = axs[1], linestyle = 'dashed', color = 'k')
axs[1].legend(loc = 'center right')
axs[1].set_xlim((-1, 2))

fig.tight_layout()

在比较 `scikit-rf` 和 `AICC` 的结果时，从时域来看，ZC 在两侧都表现出完美的匹配。

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].set_title('Absolute magnitude difference')
delta_nzc.plot_s_db(1, 0, ax = axs[0])
delta_nzc.plot_s_db(3, 2, ax = axs[0])
delta_zc.plot_s_db(1, 0, ax = axs[0])
delta_zc.plot_s_db(3, 2, ax = axs[0])
axs[0].legend(loc = 'upper right')

axs[1].set_title('Absolute phase difference')
delta_nzc.plot_s_deg(1, 0, ax = axs[1])
delta_nzc.plot_s_deg(3, 2, ax = axs[1])
delta_zc.plot_s_deg(1, 0, ax = axs[1])
delta_zc.plot_s_deg(3, 2, ax = axs[1])
axs[1].legend(loc = 'upper right')

fig.tight_layout()

对于 NZC 算法，`scikit-rf` 和 `AICC De-Embedding Utility` 的结果一致性更好，误差小于 ±0.05 dB 和 ±0.5°。对于 ZC 算法，在整个带宽范围内，误差保持在 ±0.4 dB 和 ±2° 以内，这已经足够好。总而言之，对于我们研究的特定混合模式情况，我们可以说 `scikit-rf` 在 IEEEP370 去嵌入例程中，与 `AICC De-embedding Utility` 的实现给出了 consistent 的结果。它还解决了 `AICC De-embedding Utility` 1.0.0 版本中的 NZC FIX-2 反转错误。